# Dormancy zoo: Colab launcher (Phase 4b)

Plan: `reports/2026-11-14-colab-plan.md`, approved by the PI on 2026-11-14.
**Launcher only.** This notebook holds no training code. It checks out a pinned commit, installs the
environment with pixi, and runs one queue with `python -m zoo.colab queue`. Re-running it resumes
the queue: finished runs are skipped and interrupted training resumes from `resume.pt`.

Code source (`CODE_SOURCE`):
- `"drive"` (current route, no GitHub): the code comes from `code/dormancy_zoo.bundle` in the Drive
  folder, a git bundle of the repository made on the laptop. No token is needed and nothing is pushed;
  small results are written to `results/` in the Drive folder. Open the copy of this notebook that sits
  in the Drive folder: its COMMIT is already filled in.
- `"github"`: clone from the private repository with a token from Colab secrets, push small results.

Rules (AGENTS.md, "Colab"):
- The GitHub token comes only from Colab secrets (`GITHUB_TOKEN`), through a git credential helper.
  It is never printed, never in a URL, never written to `.git/config`, never committed.
- Weights go only to Drive. Git receives only small result files (`results/colab/<queue>/`,
  `results/manifest_colab.csv`), on a feature branch.
- The first queue is the equivalence test. Nothing else runs until the PI's OK.

Copies in the Drive folder (COMMIT filled in):
- `launch_any_GPU.ipynb` (`QUEUE = "auto"`), for every session whatever GPU Colab assigns: it runs what that GPU
  may run (`configs/colab/gpu_queues.yaml`). On a validated GPU (G4, A100 40 GB, A100 80 GB) that is the per-seed distillation
  queue: G4 sessions take seeds from 0 upward, A100 sessions of either variant claim seeds from 4 downward, and every
  seed stays on one GPU model; the two A100 variants are separate models (PI 2026-11-16). Other GPUs stop.

Runs reach Drive as they finish, and an interrupted distillation's checkpoint (`resume.pt`) is copied to Drive
every few minutes, so a disconnected session loses at most a few minutes of work: just Run all again.
At startup the notebook checks that the code folder on Drive holds exactly the commit this notebook pins, and stops
with a clear message if not (PI 2026-11-16); `zoo.colab queue` repeats the check.
The GPU check (validated list; the queue's GPU lock, or for the per-seed queue the per-seed locks) always runs. Check the parameters in the next
cell, then *Runtime → Run all*. Use a GPU runtime.

In [ ]:
# ---- parameters (the agent supplies COMMIT and QUEUE) ----
CODE_SOURCE = "drive"                        # "drive": git bundle on Drive, no token; "github": private repo + token
REPO = "github.com/OWNER/REPO.git"          # github route only: without https:// and without any token
COMMIT = "PINNED_COMMIT_SHA"                 # full 40-character SHA
QUEUE = "auto"  # a queue file, or "auto": the queue for this GPU (gpu_queues.yaml)
DRIVE_DIR = "/content/drive/MyDrive/dormancy_zoo"
PIXI_VERSION = "v0.81.0"                     # same pixi as on the laptop
PIXI_MANIFEST = "pixi.toml"                  # laptop lock; a separate Colab environment only if the driver needs it
PUSH_RESULTS = True                          # github route only; the drive route never pushes


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
print(sorted(os.listdir(f"{DRIVE_DIR}/runs"))[:40])

In [ ]:
# Check out the pinned commit.
# drive route: clone from the git bundle on Drive (no token, no network access to GitHub).
# github route: the token is passed to git through GIT_ASKPASS from an environment variable of the
# child process only; output is captured so nothing can echo it.
import os, subprocess, stat
assert CODE_SOURCE in ("drive", "github"), CODE_SOURCE
assert len(COMMIT) == 40, "COMMIT must be the full SHA (use the notebook copy in the Drive folder)"

ASKPASS = "/content/.git_askpass.sh"
def git(*args, cwd="/content/dormancy_zoo", secret=False):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if secret:
        from google.colab import userdata
        with open(ASKPASS, "w") as f:
            f.write('#!/bin/sh\nprintf "%s" "$ZOO_GH_TOKEN"\n')
        os.chmod(ASKPASS, stat.S_IRWXU)
        env.update(GIT_ASKPASS=ASKPASS, ZOO_GH_TOKEN=userdata.get("GITHUB_TOKEN"))
    r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}); stderr withheld in case it holds secrets")
    return r.stdout.strip()

if CODE_SOURCE == "drive":
    BUNDLE = f"{DRIVE_DIR}/code/dormancy_zoo.bundle"
    # Code-version check (PI 2026-11-16), before anything is cloned or run: the code folder on Drive must hold
    # exactly the commit this notebook pins, both in code/COMMIT.txt and as the bundle's HEAD.
    STAMP = f"{DRIVE_DIR}/code/COMMIT.txt"
    code_commit = open(STAMP).read().strip() if os.path.exists(STAMP) else "(code/COMMIT.txt missing)"
    heads = git("bundle", "list-heads", BUNDLE, cwd="/content").splitlines() if os.path.exists(BUNDLE) else []
    bundle_head = next((ln.split()[0] for ln in heads if ln.split()[-1] == "HEAD"), "(no bundle HEAD)")
    if not (code_commit == bundle_head == COMMIT):
        raise RuntimeError(
            "STOP: CODE VERSION MISMATCH. Nothing has run.\n"
            f"  this notebook expects commit   {COMMIT[:12]}\n"
            f"  code/COMMIT.txt on Drive says  {code_commit[:12]}\n"
            f"  code/dormancy_zoo.bundle HEAD  {bundle_head[:12]}\n"
            "Upload the code folder and launch_any_GPU.ipynb from the same export (the laptop's export/drive_upload/), "
            "move older notebooks out of the Drive folder, open the new notebook, and Run all again.")
    print("code version check: notebook, code/COMMIT.txt and bundle all at", COMMIT[:12])
    if not os.path.exists("/content/dormancy_zoo/.git"):
        git("clone", "--quiet", BUNDLE, "/content/dormancy_zoo", cwd="/content")
    git("fetch", "--quiet", BUNDLE, "+refs/heads/*:refs/remotes/bundle/*")
else:
    if not os.path.exists("/content/dormancy_zoo/.git"):
        git("clone", "--quiet", f"https://x-access-token@{REPO}", "/content/dormancy_zoo", cwd="/content", secret=True)
    git("fetch", "--quiet", "origin", secret=True)
git("checkout", "--quiet", COMMIT)
head = git("rev-parse", "HEAD")
assert head == COMMIT, f"HEAD {head} is not the pinned commit"
print("at pinned commit", head, f"(from {CODE_SOURCE})")


In [ ]:
# GPU check (PI 2026-11-14): stop unless this GPU is allowed by the queue AND validated
# (configs/colab/validated_gpus.yaml; equivalence-test queues may also use candidate GPUs).
# A queue never changes GPU: the GPU of its first start is recorded on Drive and checked on resume; the per-seed
# queue (2026-11-16) records one GPU model per seed instead.
import subprocess
if QUEUE == "auto":   # any-GPU launcher: the queue this GPU may run (configs/colab/gpu_queues.yaml)
    r = subprocess.run(["python3", "-I", "src/zoo/gpucheck.py", "--pick"],
                       cwd="/content/dormancy_zoo", capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, "no queue for this GPU: choose a G4 or A100 runtime, then Run all again"
    QUEUE = r.stdout.strip().splitlines()[-1]
    print("queue for this GPU:", QUEUE)
r = subprocess.run(["python3", "-I", "src/zoo/gpucheck.py", QUEUE, "--drive", DRIVE_DIR],
                   cwd="/content/dormancy_zoo", capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, "GPU check failed (see above): choose the runtime type this queue needs, then Run all again"


In [ ]:
# Environment: pixi from the lock file, then check CUDA. If CUDA is unavailable, stop and send the
# printed driver version to the agent (a separate Colab environment would be defined; the laptop lock is never changed).
import os, subprocess
os.chdir("/content/dormancy_zoo")
if not os.path.exists(os.path.expanduser("~/.pixi/bin/pixi")):
    subprocess.run(f"curl -fsSL https://pixi.sh/install.sh | PIXI_VERSION={PIXI_VERSION} bash", shell=True, check=True)
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]
subprocess.run(["pixi", "install", "--frozen", "--manifest-path", PIXI_MANIFEST], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
r = subprocess.run(["pixi", "run", "--manifest-path", PIXI_MANIFEST, "python", "-c",
                    "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.stdout.strip().endswith("True"), "CUDA not available with this environment: stop and report the driver version above"


In [ ]:
# The PI's upload: every teacher file present on Drive and matching the laptop's hashes.
import subprocess
os.environ["ZOO_ENV"] = "colab"
subprocess.run(["pixi", "run", "--manifest-path", PIXI_MANIFEST, "python", "-m", "zoo.colab", "verify",
                f"{DRIVE_DIR}/upload_files.sha256", "--root", f"{DRIVE_DIR}/runs"], check=True)

In [ ]:
# Run the queue (resumable). Finished runs are copied to Drive and re-hashed there as they finish, and an
# in-progress distillation's resume.pt every few minutes (PI 2026-11-16).
import subprocess, sys
p = subprocess.Popen(["pixi", "run", "--manifest-path", PIXI_MANIFEST, "python", "-u", "-m", "zoo.colab", "queue",
                      QUEUE, "--drive", DRIVE_DIR], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Loading weights" not in line and "UNEXPECTED" not in line:
        sys.stdout.write(line)
assert p.wait() == 0, "queue did not finish: re-run the notebook to resume"


In [ ]:
# github route: push only the small result files to a feature branch, after guards.
# drive route: nothing is pushed; the small results are already in the Drive folder.
import os, re, pathlib, datetime
if CODE_SOURCE == "drive":
    qname = pathlib.Path(QUEUE).stem
    print(f"Results are on Drive: {DRIVE_DIR}/results/colab/{qname}/ and {DRIVE_DIR}/results/manifest_colab.csv")
    print("Weights and checkpoints: " + f"{DRIVE_DIR}/runs/. Nothing was pushed.")
elif PUSH_RESULTS:
    qname = pathlib.Path(QUEUE).stem
    branch = f"colab/{qname}-{datetime.date.today():%Y%m%d}"
    name, email = git("log", "-1", "--format=%an"), git("log", "-1", "--format=%ae")
    git("config", "user.name", name); git("config", "user.email", email)
    git("checkout", "--quiet", "-B", branch)
    git("add", f"results/colab/{qname}", "results/manifest_colab.csv")
    staged = [s for s in git("diff", "--cached", "--name-only").splitlines() if s]
    token = userdata.get("GITHUB_TOKEN")
    for s in staged:
        p = pathlib.Path(s)
        assert s.startswith(f"results/colab/{qname}/") or s == "results/manifest_colab.csv", f"refusing {s}"
        assert p.suffix in {".csv", ".json", ".yaml", ".sha256"} and p.stat().st_size <= 2 * 1024**2, f"refusing {s}"
        assert not s.startswith("runs/") and p.suffix not in {".safetensors", ".pt", ".bin"}, f"refusing {s}"
        txt = p.read_text(errors="ignore")
        assert token not in txt and not re.search(r"(ghp_|github_pat_)[A-Za-z0-9_]{20,}", txt), f"token pattern in {s}"
    del token
    git("commit", "--quiet", "-m", f"Colab results: {qname} (commit {COMMIT[:12]})")
    git("push", "--quiet", "origin", f"HEAD:refs/heads/{branch}", secret=True)
    print(f"pushed {len(staged)} files to {branch}")